# 04 — Model Training
**Objectif :** Construire et entraîner le CNN.

Architecture :
- ZeroPadding → Conv2D → BatchNorm → ReLU → MaxPool → MaxPool → Flatten → Dense(sigmoid)

Pourquoi si simple ? ResNet50 et VGG16 overfittaient sur ce petit dataset.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # CPU uniquement

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    BatchNormalization, ZeroPadding2D, Activation, Dropout
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.optimizers import Adam

print('TensorFlow version :', tf.__version__)

## 1. Charger les données préprocessées

In [ ]:
X_train = np.load('data/X_train.npy')
X_val   = np.load('data/X_val.npy')
X_test  = np.load('data/X_test.npy')
y_train = np.load('data/y_train.npy')
y_val   = np.load('data/y_val.npy')
y_test  = np.load('data/y_test.npy')

print(f'Train      : {X_train.shape}')
print(f'Validation : {X_val.shape}')
print(f'Test       : {X_test.shape}')

## 2. Architecture CNN

In [ ]:
def build_model(input_shape=(240, 240, 3)):
    model = Sequential([
        ZeroPadding2D(padding=(2, 2), input_shape=input_shape),

        Conv2D(filters=32, kernel_size=(7, 7), strides=(1, 1)),
        BatchNormalization(),
        Activation('relu'),

        MaxPooling2D(pool_size=(4, 4), strides=(4, 4)),
        Dropout(0.25),   # désactive 25% des neurones aléatoirement → réduit l'overfitting

        MaxPooling2D(pool_size=(4, 4), strides=(4, 4)),

        Flatten(),
        Dense(32, activation='relu'),
        Dropout(0.5),    # désactive 50% avant la sortie → force la généralisation
        Dense(1, activation='sigmoid')
    ])
    return model

model = build_model()
model.summary()

## 3. Compilation

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=5e-4),  # learning rate plus faible = apprentissage plus stable
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('Modèle compilé.')

## 4. Callbacks

In [ ]:
callbacks = [
    ModelCheckpoint(
        filepath='models/best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,           # on attend 8 epochs sans amélioration (plus de patience)
        restore_best_weights=True,
        verbose=1
    )
]

## 5. Entraînement

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=callbacks
)

## 6. Courbes d'entraînement

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

# Loss
axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.suptitle('Courbes d\'entraînement', fontsize=13)
plt.tight_layout()
plt.show()